# Datasets, tensor specs, and transform nodes

This standalone lesson requires the matching installed DRYML version, NumPy, and only the base DRYML installation. It uses fixed in-memory values, runs offline, and creates no files.

A current DRYML data pipeline is a composition of re-iterable dataset objects. A source yields one element at a time, its `spec` describes that one element, method nodes transform each element, and structural nodes change iteration structure.

In [ ]:
import numpy as np

from dryml.core2 import TensorSpec
from dryml.core2.cardinality import Cardinality
from dryml.data import (
    ArrayDataset,
    Batch,
    Cast,
    Flatten,
    GeneratorDataset,
    Map,
    Pipe,
    Project,
    Scale,
    Select,
    Take,
    Zip,
)


def finite_row_factory():
    """Return a fresh finite iterator each time the dataset asks."""
    for start in (2, 4, 6):
        yield np.array([start, start + 1], dtype=np.int16)


## Array sources and leading-axis cardinality

`ArrayDataset` treats the leading axis as dataset cardinality and removes that axis from the element spec. Here five stacked `2 x 2` arrays therefore yield five elements with shape `(2, 2)`. A spec may be supplied explicitly or inferred from the NumPy value.

In [ ]:
features = np.array(
    [
        [[0, 10], [20, 30]],
        [[40, 50], [60, 70]],
        [[80, 90], [100, 110]],
        [[120, 130], [140, 150]],
        [[160, 170], [180, 190]],
    ],
    dtype=np.uint8,
)
feature_spec = TensorSpec('uint8', shape=(2, 2), backend='numpy')
explicit_features = ArrayDataset(features, spec=feature_spec)
inferred_features = ArrayDataset(features)

assert len(explicit_features) == 5
assert explicit_features.peek().shape == (2, 2)
assert np.array_equal(explicit_features.peek(), features[0])
assert explicit_features.spec == feature_spec
assert inferred_features.spec == feature_spec
assert [item.tolist() for item in inferred_features] == [item.tolist() for item in inferred_features]


## Generator sources are factories, not consumed iterators

`GeneratorDataset` receives a module-level callable that creates an iterable. DRYML calls the factory for every iteration, so both passes below start from a fresh generator. The finite cardinality and element `TensorSpec` are explicit.

In [ ]:
generated = GeneratorDataset(
    finite_row_factory,
    cardinality=Cardinality.finite(3),
    spec=TensorSpec('int16', shape=(2,), backend='numpy'),
)
first_pass = [row.tolist() for row in generated]
second_pass = [row.tolist() for row in generated]

assert iter(generated) is not iter(generated)
assert first_pass == [[2, 3], [4, 5], [6, 7]]
assert second_pass == first_pass
assert generated.__len__() == Cardinality.finite(3)


## Supervised trees and method nodes

Arrays may form nested element trees when all leaves have the same leading length. This `(x, y)` source yields one image and one scalar label. `Map` applies a `Project` of method pipelines: select each branch, cast it, scale and flatten the feature, and preserve the supervised tuple structure. Each method also propagates its output spec.

In [ ]:
labels = np.array([0, 1, 0, 1, 1], dtype=np.int32)
supervised = ArrayDataset((features, labels))
assert supervised.spec == (
    TensorSpec('uint8', shape=(2, 2), backend='numpy'),
    TensorSpec('int32', shape=(), backend='numpy'),
)

prepare_sample = Project(
    Pipe(Select(0), Cast('float32'), Scale.from_range(0, 200), Flatten()),
    Pipe(Select(1), Cast('int64')),
)
prepared = Map(supervised, prepare_sample)
prepared_spec = (
    TensorSpec('float32', shape=(4,), backend='numpy'),
    TensorSpec('int64', shape=(), backend='numpy'),
)
assert prepared.spec == prepared_spec

first_x, first_y = prepared.peek()
np.testing.assert_allclose(first_x, np.array([0.0, 0.05, 0.1, 0.15], dtype=np.float32))
assert first_y.dtype == np.dtype('int64')
assert int(first_y) == 0


## Structural batching and taking

`Batch` groups elements and adds a batch axis to every spec leaf. `Take` limits the number of batches without changing the element spec. The batch spec records the configured size of two; because `drop_remainder` defaults to false, iterating all five samples also produces a final runtime batch of size one.

In [ ]:
batched = Batch(prepared, batch_size=2)
first_two_batches = Take(batched, 2)
batched_spec = (
    TensorSpec('float32', shape=(4,), batch=2, backend='numpy'),
    TensorSpec('int64', shape=(), batch=2, backend='numpy'),
)
assert batched.spec == batched_spec
assert first_two_batches.spec == batched_spec
assert batched.__len__() == Cardinality.finite(3)
assert first_two_batches.__len__() == Cardinality.finite(2)

selected = list(first_two_batches)
assert [batch_x.shape for batch_x, _ in selected] == [(2, 4), (2, 4)]
assert [batch_y.tolist() for _, batch_y in selected] == [[0, 1], [0, 1]]
np.testing.assert_allclose(selected[0][0], features[:2].reshape(2, 4).astype(np.float32) / 200)

all_batches = list(batched)
assert [batch_x.shape[0] for batch_x, _ in all_batches] == [2, 2, 1]
assert all_batches[-1][1].tolist() == [1]
np.testing.assert_allclose(all_batches[-1][0], features[4:].reshape(1, 4).astype(np.float32) / 200)


## Combining sources

Combination nodes compose existing datasets. `Zip` advances its source leaves together and exposes the same tree in its values and specs; it stops at the shortest source.

In [ ]:
sample_ids = ArrayDataset(np.arange(5, dtype=np.int64))
identified = Zip(sample=prepared, sample_id=sample_ids)
assert identified.spec == {
    'sample': prepared_spec,
    'sample_id': TensorSpec('int64', shape=(), backend='numpy'),
}
assert identified.__len__() == Cardinality.finite(5)
identified_first = identified.peek()
assert int(identified_first['sample_id']) == 0
np.testing.assert_allclose(identified_first['sample'][0], first_x)


## Deterministic source errors

An array tree is aligned only when every leaf has the same leading dimension. Construction rejects a mismatch. A zero-length source is valid and has a spec, but `peek()` cannot return an element. The current API reports both boundaries as `ValueError`; handle expected teaching failures rather than storing traceback output.

In [ ]:
try:
    ArrayDataset((np.zeros((2, 1), dtype=np.float32), np.zeros((3,), dtype=np.int64)))
except ValueError as error:
    assert 'agree on leading length' in str(error)
else:
    raise AssertionError('mismatched leading dimensions must fail')

empty = ArrayDataset(np.empty((0, 2), dtype=np.float32))
assert len(empty) == 0
assert empty.spec == TensorSpec('float32', shape=(2,), backend='numpy')
try:
    empty.peek()
except ValueError as error:
    assert str(error) == 'Cannot peek an empty dataset.'
else:
    raise AssertionError('empty peek must fail')
